# Step 5C.2 — Dedup (UNSW-NB15) + Temporal Split Boundaries + Feature Contract Audit
Не будує моделі, не фіксує остаточний split. Три задачі:
1. Видалити 480,629 повних дублікатів UNSW-NB15 ДО будь-якого спліту (зберегти дедуплікований
   parquet для повторного використання).
2. Показати фактичний часовий діапазон кожного датасету (`StartTime`/`Timestamp`/`Stime`),
   щоб обрати межі train/val/test за часом, а не випадково.
3. Перевірити feature contract `cve_ctx_*` на КОНСТАНТНІ (nunique=1) колонки окремо для
   кожного датасету — такі ознаки не несуть інформації для F1 у ЦЬОМУ конкретному датасеті
   (напр. `cve_ctx_use_epss`/`cve_ctx_service_in_kev` для CIC/UNSW).

Ядро: IDS Research.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ENRICHED_DIR = Path("/srv/ids_research/data/enriched")
DATASETS_RAW = {
    "GeNIS": dict(file="genis_with_cve_context.parquet", label_col="SubCategoryLabel",
                  benign_prefix="benign", time_col="StartTime"),
    "CIC-IDS2017": dict(file="cic_ids2017_with_cve_context.parquet", label_col="attack_label",
                        benign_value="BENIGN", time_col="Timestamp"),
    "UNSW-NB15": dict(file="unsw_nb15_with_cve_context.parquet", label_col="attack_cat",
                      benign_value="Normal", time_col="Stime"),
}
print("Ядро: IDS Research. Датасети:", list(DATASETS_RAW.keys()))

Ядро: IDS Research. Датасети: ['GeNIS', 'CIC-IDS2017', 'UNSW-NB15']


In [2]:
# ---------- 1. UNSW-NB15: видалення exact duplicate rows ДО будь-якого спліту ----------
unsw_path = ENRICHED_DIR / DATASETS_RAW["UNSW-NB15"]["file"]
unsw = pd.read_parquet(unsw_path)
n_before = len(unsw)
n_dup = unsw.duplicated().sum()
print(f"UNSW-NB15 ДО дедуплікації: {n_before:,} рядків, {n_dup:,} повних дублікатів ({n_dup/n_before:.2%})")

unsw_dedup = unsw.drop_duplicates().reset_index(drop=True)
n_after = len(unsw_dedup)
print(f"UNSW-NB15 ПІСЛЯ дедуплікації: {n_after:,} рядків (видалено {n_before - n_after:,})")

print("\nNormal/Attack ДО і ПІСЛЯ дедуплікації:")
for label, df_ in [("до", unsw), ("після", unsw_dedup)]:
    is_attack = df_["attack_cat"] != "Normal"
    n_normal, n_att = int((~is_attack).sum()), int(is_attack.sum())
    print(f"  {label}: Normal={n_normal:,} ({n_normal/len(df_):.2%})  Attack={n_att:,} ({n_att/len(df_):.2%})")

dedup_path = ENRICHED_DIR / "unsw_nb15_with_cve_context_dedup.parquet"
unsw_dedup.to_parquet(dedup_path, index=False)
print(f"\n✅ Збережено дедуплікований датасет: {dedup_path}")
del unsw   # звільняємо пам'ять, далі працюємо лише з unsw_dedup

UNSW-NB15 ДО дедуплікації: 2,540,047 рядків, 480,629 повних дублікатів (18.92%)
UNSW-NB15 ПІСЛЯ дедуплікації: 2,059,418 рядків (видалено 480,629)

Normal/Attack ДО і ПІСЛЯ дедуплікації:
  до: Normal=2,218,764 (87.35%)  Attack=321,283 (12.65%)
  після: Normal=1,959,775 (95.16%)  Attack=99,643 (4.84%)

✅ Збережено дедуплікований датасет: /srv/ids_research/data/enriched/unsw_nb15_with_cve_context_dedup.parquet


In [3]:
# ---------- 2. Repeated 5-tuple ПІСЛЯ дедуплікації UNSW (перевірка, чи змінилась картина) ----------
identity_cols_unsw = ["srcip", "dstip", "sport", "dsport", "proto"]
n_repeated_after = unsw_dedup.duplicated(subset=identity_cols_unsw).sum()
print(f"UNSW-NB15 repeated 5-tuple ПІСЛЯ дедуплікації: {n_repeated_after:,} "
      f"({n_repeated_after/len(unsw_dedup):.2%}) — це діагностика, НЕ додаткові дублікати для видалення")

UNSW-NB15 repeated 5-tuple ПІСЛЯ дедуплікації: 102,197 (4.96%) — це діагностика, НЕ додаткові дублікати для видалення


In [4]:
# ---------- 3. Реальні timestamp-межі (НЕ інтерпольовані квантилі) + class distribution ----------
DATASETS_BINARY = {
    "GeNIS": dict(label_col="SubCategoryLabel", benign_prefix="benign"),
    "CIC-IDS2017": dict(label_col="attack_label", benign_value="BENIGN"),
    "UNSW-NB15": dict(label_col="attack_cat", benign_value="Normal"),
}

loaded = {"UNSW-NB15": unsw_dedup}
for name, cfg in DATASETS_RAW.items():
    if name == "UNSW-NB15":
        continue
    loaded[name] = pd.read_parquet(ENRICHED_DIR / cfg["file"])

# binary target для КОЖНОГО датасету (потрібен для report_split нижче)
is_attack_by_name = {}
for name, bcfg in DATASETS_BINARY.items():
    df = loaded[name]
    if "benign_prefix" in bcfg:
        is_attack = ~df[bcfg["label_col"]].astype(str).str.lower().str.startswith(bcfg["benign_prefix"])
    else:
        is_attack = df[bcfg["label_col"]] != bcfg["benign_value"]
    is_attack_by_name[name] = is_attack

def report_split(name, t, boundary_1, boundary_2, is_attack):
    masks = {
        "train": t < boundary_1,
        "val": (t >= boundary_1) & (t < boundary_2),
        "test": t >= boundary_2,
    }
    print(f"\n{name}  (boundary_1={boundary_1}, boundary_2={boundary_2})")
    for split, m in masks.items():
        n = int(m.sum())
        a = int(is_attack[m.values].sum()) if n else 0
        rate = a / n if n else float("nan")
        print(f"  {split:5s}: n={n:>10,}  normal={n-a:>10,}  attack={a:>10,}  attack_rate={rate:.4%}")
    return masks

split_masks = {}
for name, cfg in DATASETS_RAW.items():
    df = loaded[name]
    time_col = cfg["time_col"]
    print(f"\n===== {name}: діапазон {time_col} =====")
    if time_col == "Stime":
        t = pd.to_datetime(df[time_col], unit="s")
    else:
        t = pd.to_datetime(df[time_col])
    print(f"  min={t.min()}  max={t.max()}  розмах={t.max() - t.min()}")

    # interpolation="nearest" гарантує, що boundary — СПОСТЕРЕЖУВАНЕ значення timestamp
    # (а не вигадане інтерпольоване число між двома точками). Але саму відсутність розрізання
    # групи однакових timestamp забезпечує НЕ interpolation сам по собі, а КОМБІНАЦІЯ:
    # observed boundary + маска "< boundary" / ">= boundary" — усі рядки з timestamp, що
    # ДОРІВНЮЄ boundary, потрапляють РАЗОМ у пізніший спліт.
    boundary_1 = t.quantile(0.70, interpolation="nearest")
    boundary_2 = t.quantile(0.85, interpolation="nearest")

    masks = report_split(name, t, boundary_1, boundary_2, is_attack_by_name[name])
    split_masks[name] = dict(t=t, boundary_1=boundary_1, boundary_2=boundary_2, masks=masks)

    # ОБОВ'ЯЗКОВИЙ assert, не print: жодна timestamp-група не повинна потрапити у два спліти.
    train_t, val_t, test_t = t[masks["train"]], t[masks["val"]], t[masks["test"]]
    if len(train_t) and len(val_t):
        assert train_t.max() < val_t.min(), \
            f"{name}: max(train_time)={train_t.max()} НЕ < min(val_time)={val_t.min()} — leakage через межу!"
    if len(val_t) and len(test_t):
        assert val_t.max() < test_t.min(), \
            f"{name}: max(val_time)={val_t.max()} НЕ < min(test_time)={test_t.min()} — leakage через межу!"
    print(f"  ✅ max(train_time) < min(val_time) < min(test_time) — перевірено assert'ом")



===== GeNIS: діапазон StartTime =====
  min=2025-02-06 09:00:44  max=2025-02-12 16:42:04  розмах=6 days 07:41:20

GeNIS  (boundary_1=2025-02-11 17:40:35, boundary_2=2025-02-11 17:41:06)
  train: n= 1,011,576  normal=    35,006  attack=   976,570  attack_rate=96.5395%
  val  : n=   211,416  normal=         0  attack=   211,416  attack_rate=100.0000%
  test : n=   249,141  normal=         0  attack=   249,141  attack_rate=100.0000%
  ✅ max(train_time) < min(val_time) < min(test_time) — перевірено assert'ом

===== CIC-IDS2017: діапазон Timestamp =====
  min=2017-07-03 01:00:01  max=2017-07-07 12:59:00  розмах=4 days 11:58:59

CIC-IDS2017  (boundary_1=2017-07-06 09:28:00, boundary_2=2017-07-07 03:23:00)
  train: n= 1,979,346  normal= 1,713,498  attack=   265,848  attack_rate=13.4311%
  val  : n=   419,191  normal=   262,470  attack=   156,721  attack_rate=37.3865%
  test : n=   429,140  normal=   295,154  attack=   133,986  attack_rate=31.2220%
  ✅ max(train_time) < min(val_time) < min(te

In [5]:
# ---------- 4. Feature contract: risk/context (для F1) vs availability/metadata (окремо) ----------
# ПРИНЦИПОВЕ РІШЕННЯ (зафіксовано, не константи як критерій): cve_ctx_* розділяється на дві
# функціонально різні групи, а не лише на CTI/EPSS/KEV/product за походженням:
#
#   RISK_CONTEXT_FEATURES   — несуть змістовний CVE/EPSS/KEV-сигнал, ідуть у ОСНОВНИЙ F1.
#   AVAILABILITY_METADATA   — описують ЧИ доступний контекст/EPSS, а НЕ сам ризик. Зокрема
#     cve_ctx_use_epss ФАКТИЧНО є gate (epss_available AND NOT epss_is_retrospective), а
#     не незалежним risk-сигналом. Подача availability-прапорців як звичайних predictors
#     ризикує дозволити моделі вивчати "чи взагалі є CVE-мапінг для цього порту", а не
#     реальний ризик — це НЕ те, що F1 має перевіряти. Залишаються поза основним F1,
#     доступні для окремого sensitivity/diagnostic аналізу за потреби.
#
# cve_ctx_service_in_kev НЕ виключається апріорі, попри те що вона dataset-specific
# (лише GeNIS): для GeNIS вона вже показала НЕ детерміновану асоціацію з target
# (Normal=34.93% True, Attack=44.44% True — Step 5C.1), а для CIC/UNSW автоматично
# виявиться константою й буде виключена саме ЦИМ механізмом (nunique<=1), а не окремим
# правилом "викинути KEV".
RISK_CONTEXT_FEATURES = [
    "cve_ctx_service", "cve_ctx_pool_size", "cve_ctx_n_epss", "cve_ctx_epss_coverage",
    "cve_ctx_service_max_epss", "cve_ctx_n_generic_os_matches",
    "cve_ctx_n_specific_product_matches", "cve_ctx_service_in_kev",
]
AVAILABILITY_METADATA_FEATURES = [
    "cve_ctx_context_available", "cve_ctx_epss_available",
    "cve_ctx_epss_is_retrospective", "cve_ctx_use_epss",
]

constant_report = {}
for name, df in loaded.items():
    print(f"\n===== {name}: константні cve_ctx_* колонки (nunique<=1) =====")
    constants = []
    for group_name, cols in [("risk/context", RISK_CONTEXT_FEATURES),
                             ("availability/metadata", AVAILABILITY_METADATA_FEATURES)]:
        for col in cols:
            if col not in df.columns:
                continue
            nu = df[col].nunique(dropna=False)
            if nu <= 1:
                constants.append(col)
                print(f"  ⚠ {col} (group: {group_name}): nunique={nu} — КОНСТАНТА для {name}")
    if not constants:
        print("  Жодної константної колонки не знайдено.")
    constant_report[name] = constants

print("\n" + "=" * 70)
print("ЗАФІКСОВАНИЙ F0/F1 feature contract (dataset-specific, після виключення констант)")
print("=" * 70)
for name, df in loaded.items():
    f1_risk = [c for c in RISK_CONTEXT_FEATURES if c in df.columns and c not in constant_report[name]]
    excluded_avail = [c for c in AVAILABILITY_METADATA_FEATURES if c in df.columns]
    print(f"\n{name}:")
    print(f"  F1 = native flow features + {f1_risk}")
    print(f"  Поза основним F1 (availability/metadata, sensitivity-only): {excluded_avail}")



===== UNSW-NB15: константні cve_ctx_* колонки (nunique<=1) =====
  ⚠ cve_ctx_n_epss (group: risk/context): nunique=1 — КОНСТАНТА для UNSW-NB15
  ⚠ cve_ctx_service_max_epss (group: risk/context): nunique=1 — КОНСТАНТА для UNSW-NB15
  ⚠ cve_ctx_service_in_kev (group: risk/context): nunique=1 — КОНСТАНТА для UNSW-NB15
  ⚠ cve_ctx_epss_available (group: availability/metadata): nunique=1 — КОНСТАНТА для UNSW-NB15
  ⚠ cve_ctx_use_epss (group: availability/metadata): nunique=1 — КОНСТАНТА для UNSW-NB15

===== GeNIS: константні cve_ctx_* колонки (nunique<=1) =====
  Жодної константної колонки не знайдено.

===== CIC-IDS2017: константні cve_ctx_* колонки (nunique<=1) =====
  ⚠ cve_ctx_service_in_kev (group: risk/context): nunique=1 — КОНСТАНТА для CIC-IDS2017
  ⚠ cve_ctx_use_epss (group: availability/metadata): nunique=1 — КОНСТАНТА для CIC-IDS2017

ЗАФІКСОВАНИЙ F0/F1 feature contract (dataset-specific, після виключення констант)

UNSW-NB15:
  F1 = native flow features + ['cve_ctx_service', 'c

In [6]:
# ---------- ПІДСУМОК Step 5C.2 ----------
print("=" * 70)
print("Зафіксовано цим кроком (без побудови моделі)")
print("=" * 70)
print(f"1. UNSW-NB15 дедуплікований: {n_before:,} -> {n_after:,} рядків, збережено в")
print(f"   {dedup_path}")
print("2. Часові поля підтверджені: GeNIS=StartTime, CIC-IDS2017=Timestamp, UNSW-NB15=Stime")
print("   (орієнтовні 70/15/15 межі за часом показані вище — ФІНАЛЬНІ межі ще НЕ зафіксовані)")
print("3. Dataset-specific константні cve_ctx_* колонки визначені вище — виключити з F1")
print("   ОКРЕМО для кожного датасету (не єдиний спільний feature set для всіх трьох)")
print("\nНаступний крок: остаточно зафіксувати межі train/val/test (за квантилями вище чи")
print("іншим принципом) і повний F1 feature contract на датасет, ПЕРЕД написанням F0/F1 notebook.")

Зафіксовано цим кроком (без побудови моделі)
1. UNSW-NB15 дедуплікований: 2,540,047 -> 2,059,418 рядків, збережено в
   /srv/ids_research/data/enriched/unsw_nb15_with_cve_context_dedup.parquet
2. Часові поля підтверджені: GeNIS=StartTime, CIC-IDS2017=Timestamp, UNSW-NB15=Stime
   (орієнтовні 70/15/15 межі за часом показані вище — ФІНАЛЬНІ межі ще НЕ зафіксовані)
3. Dataset-specific константні cve_ctx_* колонки визначені вище — виключити з F1
   ОКРЕМО для кожного датасету (не єдиний спільний feature set для всіх трьох)

Наступний крок: остаточно зафіксувати межі train/val/test (за квантилями вище чи
іншим принципом) і повний F1 feature contract на датасет, ПЕРЕД написанням F0/F1 notebook.
